# Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load Dataset

In [ ]:
df = pd.read_csv("vgsales.csv")
df.head()

# Explore Dataset

In [ ]:
print("Dataset Shape:", df.shape)

df.info()

df.describe()

# Check Missing Values

In [ ]:
df.isnull().sum()

# Select Required Columns

In [ ]:
data = df[['Publisher',
           'Platform',
           'Genre',
           'Year',
           'Global_Sales']]

data.head()

# Remove Missing Target Values

In [ ]:
data = data.dropna(subset=['Global_Sales'])

# Fill Missing Values

In [ ]:
data['Publisher'] = data['Publisher'].fillna("Unknown")
data['Platform'] = data['Platform'].fillna("Unknown")
data['Genre'] = data['Genre'].fillna("Unknown")

data['Year'] = data['Year'].fillna(data['Year'].median())

# Features and Target

In [ ]:
X = data[['Publisher',
          'Platform',
          'Genre',
          'Year']]

y = data['Global_Sales']

# Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Preprocessing

In [ ]:
categorical_features = [
    'Publisher',
    'Platform',
    'Genre'
]

numeric_features = [
    'Year'
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'num',
            SimpleImputer(strategy='median'),
            numeric_features
        )
    ]
)

# Create Model Pipeline

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Train Model

In [ ]:
model.fit(X_train, y_train)

# Predict

In [ ]:
predictions = model.predict(X_test)

predictions[:10]

# Evaluate Model

In [ ]:
mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("Mean Absolute Error :", mae)
print("Mean Squared Error  :", mse)
print("Root Mean Squared Error :", rmse)
print("R2 Score :", r2)

# Actual vs Predicted

In [ ]:
comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': predictions
})

comparison.head(20)

# Predict New Game Sales

In [ ]:
new_game = pd.DataFrame({
    'Publisher': ['Nintendo'],
    'Platform': ['Wii'],
    'Genre': ['Sports'],
    'Year': [2008]
})

prediction = model.predict(new_game)

print("Predicted Global Sales:", prediction[0], "Million Copies")

# Predict Multiple Games

In [ ]:
games = pd.DataFrame({
    'Publisher': [
        'Nintendo',
        'Ubisoft',
        'Electronic Arts',
        'Activision'
    ],
    'Platform': [
        'Wii',
        'PS4',
        'PC',
        'X360'
    ],
    'Genre': [
        'Sports',
        'Action',
        'Simulation',
        'Shooter'
    ],
    'Year': [
        2008,
        2018,
        2016,
        2012
    ]
})

games['Predicted_Global_Sales'] = model.predict(games)

games

# Feature Importance (Optional)

In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

importances = model.named_steps['regressor'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

importance_df.sort_values(by='Importance', ascending=False).head(20)

# Gradio Framework

In [ ]:
publishers = sorted(data["Publisher"].unique().tolist())

platforms = sorted(data["Platform"].unique().tolist())

genres = sorted(data["Genre"].unique().tolist())

def predict_sales(publisher, platform, genre, year):

    sample = pd.DataFrame({
        "Publisher":[publisher],
        "Platform":[platform],
        "Genre":[genre],
        "Year":[year]
    })

    prediction = model.predict(sample)[0]
    return f"{prediction:.2f} Million Units"

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
demo = gr.Interface(
    fn=predict_sales,

    inputs=[
        gr.Dropdown(
            publishers,
            label="Publisher"
        ),

        gr.Dropdown(
            platforms,
            label="Platform"
        ),

        gr.Dropdown(
            genres,
            label="Genre"
        ),

        gr.Number(
            label="Release Year",
            value=2015
        )
    ],

    outputs=gr.Textbox(
        label="Predicted Global Sales"
    ),

    title="🎮 Video Game Global Sales Prediction",

    description="""
Predict Global Sales of a Video Game using

• Publisher

• Platform

• Genre

• Release Year

Model: Random Forest Regressor
"""
)

demo.launch(share=True)